# Circle width: concept quality for w = 2, 4, 6, 8 (VOC, COCO)

In [ ]:
import sys, os, time, pickle, functools
ROOT = "/workspaces/Object-Centric-Concept-Bottlenecks"
sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
import torchvision.transforms.v2 as T
from scipy import sparse
from tqdm import tqdm
from torchvision.models.detection import maskrcnn_resnet50_fpn
import splice
import utils
from utils import generate_object_propsals, gather_object_embeddings
from data import PascalVOCDataset, CocoDataset

from eval_utils import (NUM_OBJECTS, CLASS_NAMES, vocab_idx, SYNONYMS, draw_subset, load_subset,
                        aggregate, rank_concepts, best_positions)

WIDTHS = [2, 4, 6, 8]   # px at 224×224
REF_W     = 6              # pipeline width (default)
DET       = dict(min_size=0.01, max_size=0.85, iou=0.5, min_score=0.2)
DET_BATCH = 32             
DEVICE    = "cuda"

# SpLiCE and Mask R-CNN exactly as in generate_concepts.py
splice_pipe   = splice.load("open_clip:ViT-B-32", "laion", 10000, l1_penalty=0.25, device=DEVICE, return_weights=True)
maskrcnn_pipe = maskrcnn_resnet50_fpn(pretrained=True).eval().to(DEVICE)
custom_preprocessor = T.Compose([
    T.Resize(size=224, interpolation=T.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    T.CenterCrop(size=(224, 224)),
    T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)),
])

voc_val  = PascalVOCDataset(f"{ROOT}/data/voc", year="2012", image_set="val", download=False)
coco_val = CocoDataset(f"{ROOT}/datasets/coco", train_val="val")
coco_idx = {int(fn.split(".")[0]): i for i, fn in enumerate(coco_val.image_files)}

def load_224(ds, iid):
    if ds == "voc":
        img, cat, _, _ = voc_val[int(iid)]
    else:
        img, _, cat, _ = coco_val[coco_idx[int(iid)]]
    return img, np.asarray(cat)

print("Setup OK")

In [ ]:
# Detection once per image, circle encodings for all widths 
CACHE = f"{ROOT}/red_circle_notebooks/circle_width_cache_{{ds}}.pkl"

def encode_width(image, boxes, scores, width):
    # pipeline gather_object_embeddings, only with a different circle width
    orig = utils.draw_circle_on_image
    utils.draw_circle_on_image = functools.partial(orig, width=width)
    try:
        emb, used, n = gather_object_embeddings(
            boxes, scores, DET["min_score"], NUM_OBJECTS, image, DEVICE,
            splice_pipe, 10000, custom_preprocessor, draw_boxes=False, draw_circles=True,
        )
    finally:
        utils.draw_circle_on_image = orig
    return emb[:n].cpu().numpy(), used.cpu().numpy()   


def recompute(ds, labels, stored):
    out = {"objs": {w: [] for w in WIDTHS}, "boxes": [], "num_obj": [], "max_diff_w6": [], "label_ok": []}
    ids_all = stored["img_ids"]
    t0 = time.time()
    for b in tqdm(range(0, len(ids_all), DET_BATCH), desc=ds.upper()):
        batch  = [load_224(ds, iid) for iid in ids_all[b:b + DET_BATCH]]
        images = torch.stack([img for img, _ in batch]).to(DEVICE)
        with torch.no_grad():

            boxes, scores = generate_object_propsals(images, maskrcnn_pipe, DET["min_size"], DET["max_size"], DET["iou"])
            for j, (_, cat) in enumerate(batch):
                k = b + j
                out["label_ok"].append(np.array_equal(cat > 0, labels[k] > 0))
                for w in WIDTHS:
                    emb, used = encode_width(images[j], boxes[j], scores[j], w)
                    out["objs"][w].append(sparse.csr_matrix(emb))
                out["boxes"].append(used)
                out["num_obj"].append(len(used))
                new, ref = out["objs"][REF_W][-1].toarray(), stored["circle"][k]
                if new.shape != ref.shape:
                    out["max_diff_w6"].append(np.inf)
                else:
                    out["max_diff_w6"].append(float(np.abs(new - ref).max()) if len(ref) else 0.0)
        if b == 0:
            dt = (time.time() - t0) / len(batch)
            print(f"  {dt:.2f} s/image → estimated {dt * len(ids_all) / 60:.0f} min for {ds.upper()}")
    out["num_obj"] = np.array(out["num_obj"])
    return out

print("Encoding defined")

In [ ]:
# Sum metrics and Recall@all 
def eval_sum(full, objs, labels, ds):
    names, syn = CLASS_NAMES[ds], SYNONYMS[ds]
    label_sets = {col: {vocab_idx[w] for w in syn[n]} for col, n in enumerate(names) if n is not None}
    gt_per_img = [[c for c in np.flatnonzero(l) if label_sets[c]] for l in labels]
    keep = [i for i, g in enumerate(gt_per_img) if g]
    hit10, hit_all, found, n_active = [], [], [], []
    for i in keep:
        o = objs[i].toarray() if sparse.issparse(objs[i]) else objs[i]
        s, cnt = aggregate(full[i], o)
        best = best_positions(rank_concepts(s, cnt, "sum"), gt_per_img[i], label_sets)
        found.append(any(b is not None for b in best))
        n_active.append(np.count_nonzero(cnt))
        for c, b in zip(gt_per_img[i], best):
            hit10.append(b is not None and b <= 10)
            hit_all.append(bool((s[list(label_sets[c])] > 0).any()))
    return {"N": len(keep), "Pairs": len(hit10), "Recall@10": np.mean(hit10), "Recall@all": np.mean(hit_all),
            "Found": np.mean(found), "ActiveConcepts": np.mean(n_active)}

print("Metric function defined")

In [ ]:
# Encodings (cached) results table 
CHECK = ("Recall@10", "Recall@all", "Found", "ActiveConcepts")

REC, width_rows = {}, []
for ds in ("voc", "coco"):
    rows, labels, _, _ = draw_subset(ds)
    stored = load_subset(ds, rows)
    cache = CACHE.format(ds=ds)
    if os.path.exists(cache):
        with open(cache, "rb") as fh:
            REC[ds] = pickle.load(fh)
        print(f"{ds.upper()}: cache loaded ({cache})")
    else:
        REC[ds] = recompute(ds, labels, stored)
        with open(cache, "wb") as fh:
            pickle.dump(REC[ds], fh)
    rec = REC[ds]

    variants = [("Crop", stored["crop"]), ("Circle w=6 (stored)", stored["circle"])]
    variants += [(f"Circle w={w}", rec["objs"][w]) for w in WIDTHS]
    summ = {}
    for name, objs in variants:
        summ[name] = eval_sum(stored["full"], objs, labels, ds)
        width_rows.append({"Dataset": ds.upper(), "Variant": name, **summ[name]})

    d = np.array(rec["max_diff_w6"])
    n_diff = int((rec["num_obj"] != stored["circle_num_obj"]).sum())
    same = all(np.isclose(summ[f"Circle w={REF_W}"][m], summ["Circle w=6 (stored)"][m], rtol=0, atol=1e-9) for m in CHECK)
    print(f"{ds.upper()}: labels dataset = .h5: {all(rec['label_ok'])} | object count ≠ pipeline: {n_diff}/{len(d)} images "
          f"| max |Δ| encoding w=6 (same count): {d[np.isfinite(d)].max():.2e} | mean objects {rec['num_obj'].mean():.2f}")
    print(f"      metrics w=6 recomputed = stored: {same}")

width_results = pd.DataFrame(width_rows).set_index(["Dataset", "Variant"])
display(width_results.round(3))

In [ ]:
occ_rows = []
for ds in ("voc", "coco"):
    boxes = np.vstack([b for b in REC[ds]["boxes"] if len(b)])   # 224×224 coordinates, all objects of the subset
    short = np.minimum(boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1])
    for w in WIDTHS:
        occ_rows.append({"Dataset": ds.upper(), "Width": w, "Objects": len(short),
                         "Share shorter side ≤ 2w": (short <= 2 * w).mean()})

occlusion = pd.DataFrame(occ_rows).set_index(["Dataset", "Width"])
display(occlusion.round(4))